In [1]:
!pip install dataretrieval
!pip install duckdb

# Housekeeping 

Ensure we have directories to put things in!


In [2]:
import os
# Create local directory structure if they don't exist
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/bronze", exist_ok=True)
os.makedirs("data/quarantine", exist_ok=True)
os.makedirs("data/silver", exist_ok=True)

# Create DuckDB Connection for data exploration

Lets load up the bronze dataset and do some exploration to better understand it. We need to figure out what we need to validate when we move from bronze to silver.

In [ ]:
import duckdb

path = "data/bronze/washington_stream_sites_validated.parquet"

# 1. Create a persistent connection
con = duckdb.connect()

# 2. Register the Parquet file as a virtual table/view
con.execute(f"CREATE VIEW stream_sites AS SELECT * FROM read_parquet('{path}')")

# 3. Now you can run pure, clean SQL anywhere in your notebook!
q = f"""
SELECT * FROM stream_sites
"""

# 4. Execute the query and fetch results as a DataFrame
df = con.execute(q).fetchdf()
print(df.head())

  OrganizationIdentifier           OrganizationFormalName  \
0                USGS-ID  USGS Idaho Water Science Center   
1                USGS-ID  USGS Idaho Water Science Center   
2                USGS-ID  USGS Idaho Water Science Center   
3                USGS-ID  USGS Idaho Water Science Center   
4                USGS-ID  USGS Idaho Water Science Center   

  MonitoringLocationIdentifier                      MonitoringLocationName  \
0                USGS-12392892  BLANCHARD CR AT NORTH FORK NR BLANCHARD ID   
1                USGS-12395501   AUX GAGE FOR PEND OREILLE R AT NEWPORT WA   
2                USGS-12423675                   ROCK CREEK NR ROCKFORD WA   
3                USGS-13334301          SNAKE RIVER NR ANATONE WA NEW GAGE   
4                USGS-13335250  SNAKE RIVER TRIBUTARY NO. 8 NR LEWISTON ID   

  MonitoringLocationTypeName MonitoringLocationDescriptionText  \
0                     Stream                               NaN   
1                     Stream    

# How many nulls/unknowns in the MonitoringLocationName

It looks like theres a few hundred missing names. Lets look into it further.

In [14]:
# Get the count of MonitoringLocationName values that contain 'unknown' (case-insensitive) or are NULL
# Use the classic SQL approach because I've gotten lazy and use DuckDB most of the time and keep 
# forgetting the CASE syntax
q = f"""
SELECT 
    SUM(CASE WHEN MonitoringLocationName ILIKE '%unknown%' THEN 1 ELSE 0 END) AS unknown_count,
    SUM(CASE WHEN MonitoringLocationName IS NULL THEN 1 ELSE 0 END) AS null_count,
    COUNT(*) AS total_count
FROM stream_sites
"""

df = con.execute(q).df()
print(df)

# view the unique values in the MonitoringLocationName column to see if there are any other variations of "unknown"
q = f"""
SELECT DISTINCT MonitoringLocationName
FROM stream_sites
WHERE MonitoringLocationName ILIKE '%unknown%' OR MonitoringLocationName IS NULL
""" 

unique_names = con.execute(q).df()
print(f"\nUnique names containing 'unknown' or NULL:\n{unique_names}")

# Now inspect the two case where the name is not null but contains "unknown" to see if there are any patterns in the other columns that might help us identify more of these cases
q = f"""SELECT OrganizationIdentifier, OrganizationFormalName, MonitoringLocationIdentifier, MonitoringLocationName
FROM stream_sites
WHERE MonitoringLocationName ILIKE '%unknown%' AND MonitoringLocationName IS NOT NULL
"""

unknown_sites = con.execute(q).df()
print(f"\nSites with 'unknown' in the name but not null:\n{unknown_sites}") 



   unknown_count  null_count  total_count
0            2.0       188.0        17936

Unique names containing 'unknown' or NULL:
  MonitoringLocationName
0           CASW unknown
1        PEY1   unknown 
2                    NaN

Sites with 'unknown' in the name but not null:
  OrganizationIdentifier         OrganizationFormalName  \
0                SQUAXIN  Squaxin Island Tribe (Tribal)   
1                SQUAXIN  Squaxin Island Tribe (Tribal)   

  MonitoringLocationIdentifier MonitoringLocationName  
0                 SQUAXIN-CASW           CASW unknown  
1                 SQUAXIN-PEY1        PEY1   unknown   


## Validation takeway for MonitoringLocationName  items to do the bare minimum of standardization on:

If it is missing/white space -> "UNKNOWN", if it contains the unknown string but also other charcters(e.g. "CASW unknown") -> Leave it.  

# Providers

The two providers are NWIS and STORET.

## National Water Information System (NWIS)

The [NWIS](https://www.usgs.gov/centers/new-york-water-science-center/science/nwis-usgs-data-archive) is part of the USGS's data distribution program. There are over 1.5 million sites around the country!

## STOrage and RETrieval Dashboard

The [STORET](https://www.epa.gov/waterdata/storage-and-retrieval-dashboard) is part of the EPA, and provides a number of water quality data tools.

We may end up seeing different ways of handling the data included, depending on the provider. For now, this looks good.

# Projections

How many times has this bitten me?!?! Make sure we're standardizing to a particular projection. 

In [15]:
import duckdb

stream_sites = duckdb.read_parquet(path)

# I want to use SQL queries to explore the data, so I'll install duckdb-python, which allows me to run SQL queries on pandas DataFrames in-memory.

# We want to get the unique values of the HorizontalCoordinateReferenceSystemDatumName

# So how many are we gonna find? I'm guessing at least two, for NAD83 and WGS84, but maybe more if there are some weird outliers or if some sites are using local coordinate systems or something. 
# Let's find out!

q = f"""SELECT DISTINCT HorizontalCoordinateReferenceSystemDatumName
FROM stream_sites"""

hcrfs_datum_names = duckdb.query(q).to_df()

print("\nUnique HorizontalCoordinateReferenceSystemDatumName values:")
print(f"{hcrfs_datum_names.shape[0]} unique HorizontalCoordinateReferenceSystemDatumName values found.")
for index, row in hcrfs_datum_names.iterrows():
    print(f"{index+1}. {row['HorizontalCoordinateReferenceSystemDatumName']}")



Unique HorizontalCoordinateReferenceSystemDatumName values:
6 unique HorizontalCoordinateReferenceSystemDatumName values found.
1. nan
2. OTHER
3. NAD27
4. WGS84
5. NAD83
6. UNKWN


Alright, so thats really useful! It means we can't do any backfilling of missing values for county code, HUC based on lat/lon like I'd initially tried, without standardizing.

For now, I'll make a note for our validation schema that we need to standardize the missing/'UNKNWN' to a HCRS. In Silver we can transform to a particular format. For now we just want to be consistent.

How about: 
- missing values are filled in with 'UNKWN' to be consistent with the existing pattern. 
- 'OTHER' != 'UNKWN'. There may be a further clue in the dataset. Leave it until later.

In [17]:
q = f"""SELECT DISTINCT VerticalCoordinateReferenceSystemDatumName
FROM stream_sites"""

vcrfs_datum_names = con.query(q).to_df()

print("\nUnique VerticalCoordinateReferenceSystemDatumName values:")
print(f"{vcrfs_datum_names.shape[0]} unique VerticalCoordinateReferenceSystemDatumName values found.")
for index, row in vcrfs_datum_names.iterrows():
    print(f"{index+1}. {row['VerticalCoordinateReferenceSystemDatumName']}")



Unique VerticalCoordinateReferenceSystemDatumName values:
8 unique VerticalCoordinateReferenceSystemDatumName values found.
1. nan
2. NAVD88
3. NGVD29
4. OTHER
5. UNKWN
6. SEALV
7. LTD
8. Unknown


For the VerticalCoordinateReferenceSystemDatumName, we can do a similar standardization to "UNKNWN" for "nan" and "Unknown".

# Data exploration of the sites

Before I do any further work, lets pick a single site and do some exploration. 

In [19]:
# Assuming 'con' is your DuckDB connection and 'stream_sites' is your view
# con.execute(f"CREATE VIEW IF NOT EXISTS stream_sites AS SELECT * FROM read_parquet('{path}')")

# 1. Fetch exactly ONE row using SQL
q = "SELECT * FROM stream_sites LIMIT 1"
site_0_df = con.execute(q).df()

# 2. Extract that single row as a Pandas Series to maintain attribute access
site_0 = site_0_df.iloc[0]

print(site_0)
print(f"\nExample site name: {site_0.MonitoringLocationName}")
print(f"Example site provider: {site_0.ProviderName}")

# siteid : string
# Monitoring location identified by agency code, a hyphen, and
# identification number (Example: "USGS-05586100").
site_0_id = site_0.MonitoringLocationIdentifier
print(f"Example site id: {site_0_id}")
print(f"site_0_location: {site_0.LatitudeMeasure}, {site_0.LongitudeMeasure}")

# format our characteristics for the query - they need to be separated by semicolons
# (Assuming 'characteristics', 'start_date', and 'end_date' are defined above in your notebook)
characteristics_str = ';'.join(characteristics)
print(f"Formatted characteristics for query: {characteristics_str}")

# query dates
print(f"Querying for data from {start_date} to {end_date}...")

# 3. Call the WQP API (I combined your commented-out parameters to make the query actually filter!)
result_df, result_metadata = wqp.get_results(
    siteid=site_0_id, 
    characteristicName=characteristics_str, 
    startDateLo=start_date, 
    startDateHi=end_date
)

print(f"\nResult DataFrame info:")
print(f"result_df.shape: {result_df.shape}")
print(f"\nResult metadata: {result_metadata}")

OrganizationIdentifier                                                                       USGS-ID
OrganizationFormalName                                               USGS Idaho Water Science Center
MonitoringLocationIdentifier                                                           USGS-12392892
MonitoringLocationName                                    BLANCHARD CR AT NORTH FORK NR BLANCHARD ID
MonitoringLocationTypeName                                                                    Stream
MonitoringLocationDescriptionText                                                               None
HUCEightDigitCode                                                                         17010214.0
DrainageAreaMeasure/MeasureValue                                                                 NaN
DrainageAreaMeasure/MeasureUnitCode                                                             None
ContributingDrainageAreaMeasure/MeasureValue                                               

NameError: name 'characteristics' is not defined

# Creating a Validation schema

I want some validation on our raw data before I put it into Parquet format. Bronze should be a faithful ingestion with minimal transforms, so we're going to aim for a 'least disturbance' strategy focusing on potential foot guns.

Lets ensure we have the bare minimum for later data analysis. Normally I'd speak with the analysts who will use the data, but today that is me. I'm interested in logically dividing the data by state, county, and hydrologic units.I want to be able to map the points in a dashboard. 

## Analyst Requirements
- ability to map the sites (lat/lon)
- query by the [Hydrologic Unit Codes](https://water.usgs.gov/themes/hydrologic-units/)
- query by state
- query by county

## Data Partitioning

We need to think about good features to use for data partitions when we use the Parquet format. Since we will be using a Data Lake, we will be partitioning the data files across folders (e.g. `/state=WA/`), but Apache Spark reads a particular file internally using horizontal chunks called Row Groups. 

Be nice to your clusters; practice good data hygiene. Our partitioning strategy will dictate the file sizes. If we partition too deeply, (e.g. by state, country, AND date), we'll end up with thousands of files. Parquet likes a 512MB-1GB Row Group. If its smaller than that, you're wasting resources telling Spark to build the metadata, doing the encoding/decoding, etc.

Washington has 17k sites, but many of those sites might only have a few entries! Since I don't know yet what the data file sizes are going to look like, we're just going to keep it in mind.

## Validation Actions taken here

Following some data exploration, I've found a few items to do the bare minimum of standardization on:

* "HorizontalCoordinateReferenceSystemDatumName": "nan" -> "UNKNOWN".
* "VerticalCoordinateReferenceSystemDatumName":  "nan" -> "UNKNOWN", "Unknown" -> "UNKNOWN"
* "huc_8", "county_code", "state_code" import as floats. Normalize to clean numeric strings.
* "MonitoringLocationName": missing/white space -> "UNKNOWN", if it contains the unknown string but also other charcters(e.g. "CASW unknown") -> Leave it.  


In [ ]:
import pandas as pd
import numpy as np
from typing import Any

# We use a lowercase set for ultra-fast, case-insensitive O(1) lookups.
# We removed None, np.nan, and pd.NA because pd.isna() handles those safely.
SYNONYMS_FOR_UNKNOWN = {"unknown", "unk", "unkwn", "n/a", "na", "none", "nan"}
UNKNOWN_STR = "UNKNOWN"

def is_unknown_value(v: Any) -> bool:
    """
    Check if a value is missing or represents an unknown state.
    Safely handles Pandas/Numpy nulls, empty strings, and text synonyms.
    
    Note: I ran into the np.nan issue, where np.nan != np.nan, which caused some headaches.
    """
    # 1. Safely catch actual data nulls first (None, np.nan, pd.NA)
    if pd.isna(v):
        return True
        
    # 2. If it is a string, check for empty whitespace or text synonyms
    if isinstance(v, str):
        clean_str = v.strip().lower()
        if clean_str == "" or clean_str in SYNONYMS_FOR_UNKNOWN:
            return True
            
    return False

In [ ]:
class ValidationError(Exception):
    """Custom exception for validation errors. We'll expand this later, but for now it's just a placeholder."""
    pass

from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import Optional
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime, timezone

class MonitoringSiteSchema(BaseModel):
    # Core identifiers
    org_id: str = Field(..., alias="OrganizationIdentifier")
    org_name: str = Field(..., alias="OrganizationFormalName")
    site_id: str = Field(..., alias="MonitoringLocationIdentifier")
    site_name: str = Field(..., alias="MonitoringLocationName")
    site_type: str = Field(..., alias="MonitoringLocationTypeName")
    
    # Geographic data (coerced automatically to floats)
    latitude: float = Field(..., alias="LatitudeMeasure")
    longitude: float = Field(..., alias="LongitudeMeasure")
    
    # Standardize the CRS datum names - we'll want to ensure these are consistent and handle any missing values
    hcrs_datum_names: str = Field(..., alias="HorizontalCoordinateReferenceSystemDatumName")
    vcrs_datum_names: str = Field(..., alias="VerticalCoordinateReferenceSystemDatumName")
    
    # Standardized Codes (We accept coerced strings and clean up decimal artifacts)
    state_code: str = Field(..., alias="StateCode")
    
    # There are a few missing HUC and county codes, but since we have lat/lon we can always derive those
    # later if needed. So we make these optional and allow them to be None if missing.
    # huc_8: Optional[str] = Field(None, alias="HUCEightDigitCode")
    # county_code: Optional[str] = Field(None, alias="CountyCode")
    huc_8: str = Field(..., alias="HUCEightDigitCode")
    county_code: str = Field(..., alias="CountyCode")
    
    # Optional metadata (will default to None if missing/NaN in the data)
    drainage_area: Optional[float] = Field(None, alias="DrainageAreaMeasure/MeasureValue")
    drainage_unit: Optional[str] = Field(None, alias="DrainageAreaMeasure/MeasureUnitCode")
    provider: str = Field(..., alias="ProviderName")

    # Some site names are missing or just whitespace. Let's replace those with a standard placeholder to avoid issues downstream.
    @field_validator("hcrs_datum_names", mode="before")
    @classmethod
    def heal_hcrs_datum_names(cls, v) -> str:
        # If the name is None, empty, or pandas NaN, or in the UNKNOWN_VALUES dict, replace it with a fallback
        if is_unknown_value(v):
            return UNKNOWN_STR
        return str(v).strip()

    @field_validator("vcrs_datum_names", mode="before")
    @classmethod
    def heal_vcrs_datum_names(cls, v) -> str:
        # If the name is None, empty, or pandas NaN, or in the UNKNOWN_VALUES dict, replace it with a fallback
        if is_unknown_value(v):
            return UNKNOWN_STR
        return str(v).strip()

    # Clean up numeric codes that Pandas converted to float strings (e.g., '17010214.0' -> '17010214')
    @field_validator("huc_8", "county_code", "state_code", mode="before")
    @classmethod
    def clean_numeric_code_strings(cls, v) -> str:
        if v is None:
            return v
        
        # Coerce to a raw string if it is an int/float
        val_str = str(v).strip()
        
        # If it ends in '.0' (Pandas float inference artifact), strip it
        if val_str.endswith(".0"):
            val_str = val_str[:-2]
            
        return val_str
    
    # Some site names are missing or just whitespace. Let's replace those with a standard placeholder to avoid issues downstream.
    @field_validator("site_name", mode="before")
    @classmethod
    def heal_empty_site_names(cls, v) -> str:
        # If the name is None, empty, or pandas NaN, replace it with a fallback
        if is_unknown_value(v):
            return UNKNOWN_STR
        return str(v).strip()

    # Coordinate validation to ensure lat/lon are within valid ranges
    @field_validator("latitude")
    @classmethod
    def validate_latitude(cls, v: float) -> float:
        if not (-90.0 <= v <= 90.0):
            raise ValueError(f"Latitude {v} is out of bounds.")
        return v

    @field_validator("longitude")
    @classmethod
    def validate_longitude(cls, v: float) -> float:
        if not (-180.0 <= v <= 180.0):
            raise ValueError(f"Longitude {v} is out of bounds.")
        return v
    
    # have to set coercion to string for these code fields because WQP uses strings to represent codes (to preserve leading zeros), but the raw data may have them as numbers, which causes validation errors.
    # CountyCode
    #   Input should be a valid string [type=string_type, input_value=63.0, input_type=float]
        
    model_config = {
        "populate_by_name": True, # Allows instantiating with either the field name or its alias
        "arbitrary_types_allowed": True,
        "coerce_numbers_to_str": True 
    }

In [ ]:


# Initialize our tracking lists and telemetry counters
validated_records = []
quarantined_records = []

# Convert DataFrame to list of dicts to process. 
# Replacing NaN with None makes it compatible with Pydantic's Optional types.
raw_records = stream_sites.replace({np.nan: None}).to_dict(orient="records")

print(f"Starting validation on {len(raw_records)} records...")
start_time = datetime.now()

for idx, record in enumerate(raw_records):
    try:
        # Pass the raw record dict directly to Pydantic
        validated_model = MonitoringSiteSchema(**record)
        
        # Serialize back to dictionary using model_dump() (using real field names, not aliases)
        clean_dict = validated_model.model_dump()
        
        # Add technical metadata for Bronze layer
        clean_dict["ingestion_timestamp"] = datetime.now(timezone.utc).isoformat()
        
        validated_records.append(clean_dict)
        
    except ValidationError as e:
        # capture the exact failure telemetry
        error_details = e.errors()
        
        quarantined_entry = {
            "index": idx,
            "raw_record": record,
            "validation_errors": [
                {
                    "field": " -> ".join(map(str, err["loc"])),
                    "error_message": err["msg"],
                    "type": err["type"]
                }
                for err in error_details
            ],
            "quarantined_at": datetime.now(tz=timezone.utc).isoformat()
        }
        quarantined_records.append(quarantined_entry)

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

# Print pipeline telemetry metrics
print("\n--- INGESTION TELEMETRY METRICS ---")
print(f"Processing Time   : {duration:.2f} seconds")
print(f"Total Processed   : {len(raw_records)}")
print(f"Passed validation : {len(validated_records)} ({len(validated_records)/len(raw_records)*100:.2f}%)")
print(f"Quarantined (Bad) : {len(quarantined_records)} ({len(quarantined_records)/len(raw_records)*100:.2f}%)")
print("----------------------------------")

# Write out the validated records to the Bronze layer as Parquet
bronze_path = f"data/bronze/{state_name.lower().replace(' ', '_')}_stream_sites_validated.parquet"
pd.DataFrame(validated_records).to_parquet(bronze_path, index=False)
print(f"Saved validated records to Bronze layer: {bronze_path}")

# Write out the quarantined records to a JSON file for later analysis
quarantine_path = f"data/quarantine/{state_name.lower().replace(' ', '_')}_stream_sites_quarantined.json"
with open(quarantine_path, "w") as f:
    json.dump(quarantined_records, f, indent=4)
print(f"Saved quarantined records to: {quarantine_path}")


## Quarantine Zone

What went wrong? Lets figure it out. 

In [ ]:
import json
import pandas as pd

# Load the quarantined records
with open(quarantine_path, "r") as f:
    bad_records = json.load(f)

# Flatten the nested errors into a Pandas DataFrame for easy analysis
error_summary = []
for entry in bad_records:
    for err in entry["validation_errors"]:
        error_summary.append({
            "site_id": entry["raw_record"].get("MonitoringLocationIdentifier"),
            "failed_field": err["field"],
            "error_msg": err["error_message"],
            "raw_value": entry["raw_record"].get(err["field"].split(" -> ")[-1])
        })

err_df = pd.DataFrame(error_summary)

print(f"Total Unique Errors: {len(err_df)}")
print("\n--- ERROR TYPE BREAKDOWN ---")
print(err_df["failed_field"].value_counts())

print("\n--- SAMPLE ERROR ENTRIES ---")
for unique_field in err_df["failed_field"].unique():
    print(f"\nErrors for field: {unique_field}")
    sample_errors = err_df[err_df["failed_field"] == unique_field].iloc[:1]  # Get the first error for this field
    print(sample_errors[["site_id", "error_msg", "raw_value"]])


# Best Effort first pass

First pass we had only one type of ValidationError, which was failed_field.

```
Total Unique Errors: 206
--- ERROR TYPE BREAKDOWN ---
failed_field 
MonitoringLocationName 188
LatitudeMeasure 7
LongitudeMeasure 7
HUCEightDigitCode 2
CountyCode 2
``` 

## Backfilling Failed Fields

We may be able to backfill some of these fields, depending on what other information we have. Lets define a template for how to handle these. Each failed_field category should have a few key bits of data to document how we handle it. 

* Document the FieldName, e.g. 'MonitoringLocationName'
* Describe the field and our thoughts.
* If we make a change to our schema, lets document it here as 'Change'.
* Document the Verdict as 'Accept and Postpone' to represent data we add to Bronze, or 'Reject' from the dataset.
* 'Notes' for any notes on future plans, etc. 


### FieldName: MonitoringLocationName
Description: It is okay for this to be blank, as long as we have other data to map it. MonitoringLocationIdentifier,MonitoringLocationName
Change: Adjusted schema to specifically handle missing field as 'UNKNOWN_LOCATION_NAME' for later possible backfilling.
Verdict: Accept and Postpone.

In [ ]:
!pip install duckdb
import duckdb

path = "data/bronze/washington_stream_sites_validated.parquet"
con = duckdb.connect()

# I thought this read the parqet into memory, but its only a reference to the file. 
# The actual reading is happening when we run the SQL queries against it, which is why we can 
# query it without loading the whole thing into memory at once.
stream_sites = duckdb.read_parquet(path)

# match for "UNKNOWN LOCATION NAME", "unknown", "UNKNOWN", etc. to see all the records that had 
# missing/invalid site names that were healed during validation as well as possible non-standard 
# placeholders that may have slipped through.
q = f"""
SELECT site_id, site_name, latitude, longitude
FROM stream_sites
WHERE site_name ILIKE '%unknown%'
ORDER BY site_id
LIMIT 100
"""
df = con.execute(q).df()
print(df.shape)
print(df.head())


# show the unique site_names to see if we have any variation of the placeholder that we missed in our validation logic
q = f"""
SELECT DISTINCT site_name
FROM stream_sites
WHERE site_name ILIKE '%unknown%'
"""
unique_site_names = con.execute(q).df()
print("Unique site_name values for records with 'unknown' in the name:")
print(unique_site_names)


Very interesting! There are some other placeholders in there. We can do some standardization in Silver. 

Now, do we have any occurences where we have a missing MonitoringLocationName and ALSO have a missing MonitoringLocationIdentifier? Lets start by querying for missing or unknown site id.

In [ ]:
q = f"""SELECT site_id, site_name
FROM stream_sites
WHERE site_name ILIKE '%unknown%'
ORDER BY site_id
LIMIT 100"""

df = con.execute(q).df()
print(df.shape)
print(df)

In [ ]:

# Let's read in the Bronze parquet file, and query for the records that had the placeholder site name "UNKNOWN LOCATION NAME". 
# This will allow us to identify which records had missing/invalid site names that were healed during validation, 
# and we can analyze those further to see if we can backfill the missing information using geocoding or other methods.
# Or if they have valid MonitoringLocationIdentifier, do we even care?
bronze_path = "data/bronze/washington_stream_sites_validated.parquet"

# Best-practice read: project only needed columns + predicate pushdown (pyarrow)
df = pd.read_parquet(
    bronze_path,
    engine="pyarrow",
    columns=["site_id", "site_name", "latitude", "longitude"],
    filters=[("site_name", "==", UNKNOWN_VALUES["site_name"]), ("site_id", "!=", None)]
)

print(df.shape)
print(df.head())

Alright, so we have no occasions where the MonitoringLocationName is blank AND there is also a missing MonitoringLocationIdentifier. Just accept this and move on.


### FieldName: LatitudeMeasure or LongitudeMeasure
Description: The coordinates. Often both missing if one is NULL.
Verdict: Reject.
Notes: Can possibly get a rough estimate later if we have a valid MonitoringLocationName, or even join with other data to get it. It is important to understand how much effort you're going to go through for a minimal payout. Lets look at two examples:



'USGS-12062510' on [waterdata.usgs.gov](https://waterdata.usgs.gov/monitoring-location/USGS-12062510/#dataTypeId=measurements-00060-0&period=periodOfRecord), I found valid coordinates. We just don't have the correct dataset to cross-reference right now.

[USGS-1203950350](https://waterdata.usgs.gov/monitoring-location/USGS-1203950350/#dataTypeId=measurements-00060-0&period=P1Y), MonitoringLocationName="BOULDER CREEK NEAR MOUTH NEAR AMANDA PARK, WA" did not have valid coordinates. So the dataset wouldn't help. We would have to do some other form of interpolation.

Possible Backfill Options:
If we later want to backfill, query the number of measurements for each site to see if its even worth it. This example only has 1 datapoint from 2018, so it is likely not of interest for most of our users.



### HUC-8 Codes

If your [HUC-8 code](https://nas.er.usgs.gov/hucs.aspx) is missing, it should be relatively easy to backfill it later with some Pro-GIS Engineer work. Accept and Postpone.



### County Code

Same deal as the HUC-8 codes. A real Pro-GIS maverick is gonna love handling that. Accept and Postpone.

There  I marked them optional, with a default value of None. Since we have the rest of the data, we can backfill if its worth it for the few that are missing.

# Backfilling Exploration

For the three missing fields, there are several methods we can try to backfill the data. 

Case: Missing County Code
Requires: Latitude & Longitude
Process: If we have latitude and longitude, we can use geocoding to retrieve an approximate address and use the state and county names to match with the Census data, and pull the five-digit county FIPS code from there.

Case: Missing HUC_8


In [ ]:
# hahahah postpone...
!pip install geopy

import pandas as pd

# Create a mapping dictionary for state names to 2-letter state abbreviations
# (Census uses 2-letter state abbreviations like 'WA', 'AL' in the 'State' column)
state_to_abbr = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA',
    'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH',
    'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY', 'North Carolina': 'NC',
    'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA',
    'Rhode Island': 'RI', 'South Carolina': 'SC', 'South Dakota': 'SD', 'Tennessee': 'TN',
    'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA',
    'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY'
}

def normalize_name(name_str):
    """ Normalize county names by lowercasing, stripping whitespace, and removing common administrative terms."""
    if not name_str:
        return ""

    normalized = str(name_str).lower().strip()
    normalized = normalized.replace("county", "").replace("parish", "").strip()
    return normalized

# Load the FIPS table
url = "https://www2.census.gov/geo/docs/reference/codes/files/national_county.txt"
fips_df = pd.read_csv(url, header=None, names=["State", "StateFIPS", "CountyFIPS", "CountyName", "CLASSFP"], dtype=str)
fips_df['FIPS'] = fips_df['StateFIPS'] + fips_df['CountyFIPS']

fips_df['normalized_county'] = fips_df['CountyName'].apply(normalize_name)
fips_df['normalized_state'] = fips_df['State'].apply(lambda x: str(x).lower().strip())


# Apply normalization for matching keys
fips_df['normalized_county'] = fips_df['CountyName'].apply(normalize_name)
fips_df['normalized_state'] = fips_df['State'].apply(lambda x: str(x).lower().strip())

# Save this cleaned lookup index locally
fips_df.to_csv("data/raw/fips_lookup_clean.csv", index=False)


In [ ]:

def resolve_fips(raw_state, raw_county, lookup_df=fips_df):
    """
    Takes full/abbreviated state name and raw county name, 
    normalizes them, and extracts the 5-digit FIPS code.
    """
    if not raw_state or not raw_county:
        return None
        
    # Get the 2-letter state abbreviation if full name is provided
    state_abbr = state_to_abbr.get(raw_state, raw_state).lower().strip()
    norm_county = normalize_name(raw_county)
    
    # Query the normalized FIPS table
    match = lookup_df[
        (lookup_df['normalized_state'] == state_abbr) & 
        (lookup_df['normalized_county'] == norm_county)
    ]
    
    if not match.empty:
        # Return the 5-digit FIPS code
        return match['FIPS'].values[0]
    return None

# Example usage:
fips_code = resolve_fips("Washington", "King County")
print(f"Resolved FIPS code: {fips_code} == 53033")

# and for skagit county, WA
fips_code = resolve_fips("Washington", "Skagit County")
print(f"Resolved FIPS code: {fips_code} == 53057")

In [ ]:
from geopy.geocoders import Nominatim
import us # for getting the fips code for the state, once we have the county name, which we can derive from the lat/lon if needed.
import time


# Found a tutorial for the geocoders bit: https://www.geeksforgeeks.org/python/get-the-city-state-and-country-names-from-latitude-and-longitude-using-python/
# Second part is using the us package to get the FIPS code for the state, once we have the county name, which we can derive from the lat/lon if needed. 

# initialize Nominatim API 
geolocator = Nominatim(user_agent="county_finder")
successful_backfills = []

# Rate-limit Nominatim requests to avoid hitting block limits (1 second per request is polite)
print(f"Starting spatial backfill process for {len(quarantined_records)} records...\n")

for bad_record in quarantined_records:
    site_id = bad_record['raw_record'].get('MonitoringLocationIdentifier')
    
    lat = bad_record['raw_record'].get('LatitudeMeasure')
    lon = bad_record['raw_record'].get('LongitudeMeasure')
    
    if lat is None or lon is None:
        print(f"[Unrecoverable] Site {site_id} is missing coordinates. Cannot backfill.")
        continue
    
    # I need lon west, so I need to make the longitude negative.
    lon = -lon if lon > 0 else lon

    try:
        print(f"Geocoding Site {site_id} ({lat}, {lon})...")
        location = geolocator.reverse(f"{lat}, {lon}", exactly_one=True)
        
        # Example: Skagit County, Washington, United States
        # We can parse the location.raw['address'] dictionary to extract the county, state, and country information.
        
        if location and 'address' in location.raw:
            address = location.raw['address']
            county_name = address.get('county')
            state_name = address.get('state')
            
            # Use our standardized matching technique to resolve the FIPS code from the derived county and state names
            fips_5_digit = resolve_fips(state_name, county_name)
            
            if fips_5_digit:
                # WQP separates FIPS into state_code (2 digits) and county_code (3 digits)
                derived_state_code = fips_5_digit[:2]
                derived_county_code = fips_5_digit[2:]
                
                successful_backfills.append({
                    "site_id": site_id,
                    "site_name": bad_record['raw_record'].get('MonitoringLocationName') or "UNKNOWN LOCATION NAME",
                    "lat": lat,
                    "lon": lon,
                    "county": county_name,
                    "state": state_name,
                    "state_code": derived_state_code,
                    "county_code": derived_county_code,
                    "huc_8": "UNKNOWN",  # Or handle watersheds down the road
                    "source_record": bad_record['raw_record']
                })
                print(f"Successfully resolved FIPS: {fips_5_digit} (State: {derived_state_code}, County: {derived_county_code})")
            else:
                print(f"Could not resolve FIPS for State: {state_name}, County: {county_name}")
        else:
            print(f"Geocoding returned no address data for {site_id}")
            
        # Nominatim asks for max 1 request per second in their Terms of Service
        time.sleep(1.0)
    except Exception as e:
            print(f"Error geocoding site {site_id}: {e}")
            time.sleep(1.0)

    print(f"\nCompleted! Successfully backfilled {len(successful_backfills)} out of {len(quarantined_records)} quarantined records.")

for backfill in successful_backfills:
    print(f"Backfilled Site ID: {backfill['site_id']}")
    print(f"Derived County: {backfill['county']}")
    print(f"Derived State: {backfill['state']}")
    print(f"Derived County Code: {backfill['county_code']}")
    print(f"Derived HUC-8 Code: {backfill['huc_8']}")
    print("\n")
